## Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

## Load Metadata

This block loads the metadata CSV which contains:

- track IDs

- paths to the audio

- paths to mel spectrograms

- multi-label genre columns

In [ ]:
meta_path = "NLP-mini-project/datasets/fma/fma_clean/clean_metadata.csv"
df = pd.read_csv(meta_path)

print("Metadata rows:", len(df))
print(df.head())

## PyTorch Dataset for Mel Spectrograms (Multi-Label)

For each sample, it:

- loads the mel .npy file

- converts it into a PyTorch tensor

- adds a channel dimension (1 × 128 × T)

- extracts all label columns (label_Rock, etc.)

- returns (mel_tensor, label_tensor)

In [ ]:
class FMAAudioDataset(Dataset):
    """
    Loads one mel spectrogram and its multi-hot genre labels.
    """

    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

        # All label columns: label_Rock, label_Pop, ...
        self.label_cols = [c for c in df.columns if c.startswith("label_")]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # -------- Load Mel Spectrogram --------
        mel_path = row["mel_path"]  # full correct path from metadata
        mel = np.load(mel_path)     # shape: (128, T)

        mel = torch.tensor(mel, dtype=torch.float32)
        mel = mel.unsqueeze(0)       # shape: (1, 128, T)

        # -------- Load Multi-Label Vector --------
        labels = row[self.label_cols].values.astype(np.float32)
        labels = torch.tensor(labels)

        # -------- Optional Transform --------
        if self.transform:
            mel = self.transform(mel)

        return mel, labels


## Train/Validation Split

splits metadata into:

- training data (90%)

- validation data (10%)

Stratifying using the label columns to maintain genre distribution

In [ ]:
train_df, valid_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=df[[c for c in df.columns if c.startswith("label_")]]
)

## Build Dataset Objects

Creates PyTorch dataset objects

In [ ]:
train_ds = FMAAudioDataset(train_df)
valid_ds = FMAAudioDataset(valid_df)

## DataLoaders (parallelized)

The dataloaders:

- load samples in batches

- run workers in parallel

- shuffle the training data

In [ ]:
train_loader = DataLoader(
    train_ds,
    batch_size=16,
    shuffle=True,
    num_workers=16,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=16,
    shuffle=False,
    num_workers=16,
    pin_memory=True
)